# Week 3 – Data Acquisition with the ONS API
**MOD008909 Big Data Analytics · E010-2 support**

In this notebook you will **find and download the two datasets for your assignment yourself**, using the Office for National Statistics (ONS) API:

1. **Life expectancy by local authority**
2. **Gross Disposable Household Income (GDHI) by local authority**

You will not be given download links. Instead you will use two ONS APIs step by step, exactly as a data engineer would.

This is the **Acquisition** stage from the lecture: *find → examine → download → first look*. Cleaning and loading into MongoDB (ETL) is next week.

## What is an API?
An **API** (Application Programming Interface) lets a program ask a website for data.

- We send a **request** to a web address (an *endpoint*).
- The server sends back a **response**, usually in **JSON**: text made of keys and values, like a Python dictionary.

The ONS API is free and needs no key. Its base address is:

`https://api.beta.ons.gov.uk/v1`

We will travel down this path:

**datasets → one dataset → edition → version → dimensions → downloads**

In [3]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import json
import pprint
import requests

BASE = "https://api.beta.ons.gov.uk/v1"
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)                  # folder for our downloads
print("Ready. Files will be saved in:", os.path.abspath(DATA_DIR))

Ready. Files will be saved in: c:\Users\sk9131\AppData\Local\Development\BigDataAnalytics\data


## Step 1 – Your first request
We ask the API for its list of datasets.

- `requests.get(...)` sends the request
- `raise_for_status()` stops with an error if something went wrong
- `.json()` turns the JSON text into a Python dictionary

In [5]:
url = f"{BASE}/datasets"
r = requests.get(url,  timeout=30)
r.raise_for_status()

print("Status code:", r.status_code)      # 200 means OK
data = r.json()
print("Top-level keys:", list(data.keys()))
print("Datasets on this page:", data["count"], "| Total datasets:", data["total_count"])

Status code: 200
Top-level keys: ['items', 'count', 'offset', 'limit', 'total_count']
Datasets on this page: 20 | Total datasets: 338


The response is a dictionary. The datasets are inside `items`. Let's look at **one** dataset so the output stays readable.

In [4]:
pprint.pp(data["items"][0])

{'contacts': [{'email': 'qualityoflife@ons.gov.uk',
               'name': 'Will Shufflebottom, Owain Birrell, Pavan Bains and '
                       'Geeta Kerai',
               'telephone': '+44 300 0671543'}],
 'description': 'Seasonally and non seasonally-adjusted quarterly estimates of '
                'life satisfaction, feeling that the things done in life are '
                'worthwhile, happiness and anxiety in the UK.',
 'keywords': ['well-being'],
 'id': 'wellbeing-quarterly',
 'last_updated': '2023-12-13T09:40:24.204Z',
 'links': {'editions': {'href': 'https://api.beta.ons.gov.uk/v1/datasets/wellbeing-quarterly/editions'},
           'latest_version': {'href': 'https://api.beta.ons.gov.uk/v1/datasets/wellbeing-quarterly/editions/time-series/versions/9',
                              'id': '9'},
           'self': {'href': 'https://api.beta.ons.gov.uk/v1/datasets/wellbeing-quarterly'},
           'taxonomy': {'href': 'https://api.beta.ons.gov.uk/v1/peoplepopulationandc

**Look for:**
- `id` – the short name we use in later requests
- `title` and `description` – what the dataset is
- `links` → `latest_version` → `href` – the address of the newest version

✏️ *Question:* why does the API only return some datasets, not all of them? (Hint: compare `count` and `total_count`.)

## Step 2 – Get the whole catalogue (paging)
The API returns datasets in **pages**. We use `limit` (how many per page) and `offset` (where to start) and keep asking until we have them all.

In [5]:
def list_datasets(limit=100, max_pages=20):
    items, offset = [], 0
    for _ in range(max_pages):
        params = {"limit": limit, "offset": offset}
        r = requests.get(f"{BASE}/datasets", params=params, timeout=30)
        r.raise_for_status()
        j = r.json()
        items.extend(j.get("items", []))
        offset += limit
        if offset >= j.get("total_count", 0):
            break
    return items

datasets = list_datasets()
print("Datasets loaded:", len(datasets))

Datasets loaded: 338


A table is easier to read than raw JSON, so we put the key fields into a pandas DataFrame.

In [6]:
import pandas as pd
catalogue = pd.DataFrame([{
    "id": d.get("id"),
    "title": d.get("title"),
    "last_updated": d.get("last_updated"),
    "release_frequency": d.get("release_frequency"),
    "latest_version_href": (d.get("links", {}).get("latest_version") or {}).get("href"),
} for d in datasets])

pd.set_option("display.max_colwidth", 80)
catalogue.head(10)

,id,title,last_updated,release_frequency,latest_version_href
0,wellbeing-quarterly,Quarterly personal well-being estimates,2023-12-13T09:40:24.204Z,Quarterly,https://api.beta.ons.gov.uk/v1/datasets/wellbeing-quarterly/editions/time-se...
1,wellbeing-local-authority,Personal well-being estimates by local authority,2023-12-13T09:40:21.928Z,Annual,https://api.beta.ons.gov.uk/v1/datasets/wellbeing-local-authority/editions/t...
2,weekly-deaths-region,Deaths registered weekly in England and Wales by region,2026-07-01T10:46:27.849Z,Weekly,https://api.beta.ons.gov.uk/v1/datasets/weekly-deaths-region/editions/time-s...
3,weekly-deaths-local-authority,Death registrations and occurrences by local authority and place of death,2024-01-09T10:10:57.755Z,Weekly,https://api.beta.ons.gov.uk/v1/datasets/weekly-deaths-local-authority/editio...
4,weekly-deaths-health-board,Death registrations and occurrences by health board and place of death,2024-01-09T10:11:08.441Z,Weekly,https://api.beta.ons.gov.uk/v1/datasets/weekly-deaths-health-board/editions/...
5,weekly-deaths-age-sex,Deaths registered weekly in England and Wales by age and sex,2026-07-01T10:46:24.825Z,Weekly,https://api.beta.ons.gov.uk/v1/datasets/weekly-deaths-age-sex/editions/2026/...
6,uk-spending-on-cards,UK spending on credit and debit cards,2024-05-16T09:00:13.637Z,Weekly,https://api.beta.ons.gov.uk/v1/datasets/uk-spending-on-cards/editions/time-s...
7,uk-business-by-enterprises-and-local-units,"UK Business: Activity, Size and Location",2022-11-03T09:50:09.932Z,Annual,https://api.beta.ons.gov.uk/v1/datasets/uk-business-by-enterprises-and-local...
8,traffic-camera-activity,Traffic Camera Activity,2024-06-20T09:00:45.082Z,Weekly,https://api.beta.ons.gov.uk/v1/datasets/traffic-camera-activity/editions/tim...
9,trade,Trade in goods: country by commodity,2026-02-19T09:48:58.687Z,Monthly,https://api.beta.ons.gov.uk/v1/datasets/trade/editions/time-series/versions/65


## Step 3 – Search the catalogue
Now we search for our two topics by keyword.

In [7]:
def search_catalogue(df, keyword):
    k = keyword.lower()
    mask = (df["title"].fillna("").str.lower().str.contains(k)
            | df["id"].fillna("").str.lower().str.contains(k))
    return df[mask][["id", "title", "last_updated"]]

search_catalogue(catalogue, "life expectancy")

,id,title,last_updated
28,life-expectancy-by-local-authority,Life Expectancy by Local Authority,2020-12-16T13:05:35.252Z


In [8]:
search_catalogue(catalogue, "household income")

,id,title,last_updated
10,tax-benefits-statistics,Effects of Taxes and Benefits on Household Income,2022-09-16T06:15:01.401Z


✏️ *What did you find?*

- **Life expectancy by local authority** is in this catalogue. ✅
- **GDHI** is probably **not** here. This catalogue only contains ONS "Customise my data" datasets. GDHI is published through a different ONS service, so we will use a second API for it in Step 7.

This is a real acquisition lesson: **data for one project often lives in different places and formats.**

## Step 4 – Examine the life expectancy dataset
We take the dataset `id` and ask for its details, then follow the link to its **latest version**.

In [9]:
DATASET_ID = "life-expectancy-by-local-authority"

r = requests.get(f"{BASE}/datasets/{DATASET_ID}",timeout=30)
r.raise_for_status()
dataset = r.json()

print("Title:      ", dataset.get("title"))
print("Description:", dataset.get("description"))
version_href = dataset["links"]["latest_version"]["href"]
print("Latest version:", version_href)

Title:       Life Expectancy by Local Authority
Description: Subnational trends in the average number of years people will live beyond their current age measured by “period life expectancy”.
Latest version: https://api.beta.ons.gov.uk/v1/datasets/life-expectancy-by-local-authority/editions/time-series/versions/1


Now open that version. This is the most useful response: it lists the **dimensions** and the **downloads**.

In [10]:
r = requests.get(version_href,  timeout=30)
r.raise_for_status()
version = r.json()

print("Edition:", version.get("edition"), "| Version:", version.get("version"),
      "| Released:", version.get("release_date"))
print("\nDimensions:")
for d in version.get("dimensions", []):
    print(f"  - {d['name']:<12} ({d.get('label')})")

Edition: time-series | Version: 1 | Released: 2020-11-09T00:00:00.000Z

Dimensions:
  - time         (Time)
  - geography    (Geography)
  - sex          (Sex)
  - agegroups    (Age group)


**Dimensions!** These are the same idea as in the lecture: the descriptive categories the numbers are organised by (time, geography, sex, age group).

Let's look at the **options** inside some dimensions.

In [11]:
def list_dimension_options(version_href, dimension_name, limit=1000):
    url = f"{version_href}/dimensions/{dimension_name}/options"
    out, params = [], {"limit": limit, "offset": 0}
    while True:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        j = r.json()
        out += j.get("items", [])
        params["offset"] += limit
        if params["offset"] >= j.get("total_count", 0):
            break
    return out

sex_opts = list_dimension_options(version_href, "sex")
print("Sex options:", [(o["label"], o["option"]) for o in sex_opts])

time_opts = list_dimension_options(version_href, "time")
print("Time options (first 10):", [o["label"] for o in time_opts[:10]])

geo_opts = list_dimension_options(version_href, "geography")
print("Number of geography options:", len(geo_opts))
print("Examples:", [(o["label"], o["option"]) for o in geo_opts[:5]])

Sex options: [('Female', 'female'), ('Male', 'male')]
Time options (first 10): ['2001-03', '2002-04', '2003-05', '2004-06', '2005-07', '2006-08', '2007-09', '2008-10', '2009-11', '2010-12']
Number of geography options: 421
Examples: [('Hartlepool', 'E06000001'), ('Middlesbrough', 'E06000002'), ('Redcar and Cleveland', 'E06000003'), ('Stockton-on-Tees', 'E06000004'), ('Darlington', 'E06000005')]


✏️ *Questions:*
1. What is the time unit? (Single years or periods?)
2. What do geography codes look like? (e.g. `E06…`, `E07…`) We will need these codes to **join** with GDHI later.

## Step 5 – Find the download link through the API
The version response contains a `downloads` section. We **read** the CSV address from it rather than typing it in.

In [12]:
downloads = version.get("downloads", {})
for fmt, info in downloads.items():
    print(f"{fmt:5} size: {int(info.get('size', 0))/1_000_000:6.1f} MB  ->  {info.get('href')}")

life_csv_url = downloads["csv"]["href"]

xls   size:    4.9 MB  ->  https://download.ons.gov.uk/downloads/datasets/life-expectancy-by-local-authority/editions/time-series/versions/1.xlsx
csv   size:   22.7 MB  ->  https://download.ons.gov.uk/downloads/datasets/life-expectancy-by-local-authority/editions/time-series/versions/1.csv
csvw  size:    0.0 MB  ->  https://download.ons.gov.uk/downloads/datasets/life-expectancy-by-local-authority/editions/time-series/versions/1.csv-metadata.json


Now download the file. We use `stream=True` so a large file is saved in chunks.

In [13]:
def download_file(url, path):
    with requests.get(url,  stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    print("Saved:", path, f"({os.path.getsize(path)/1_000_000:.1f} MB)")

LIFE_PATH = os.path.join(DATA_DIR, "life_expectancy_by_local_authority.csv")
download_file(life_csv_url, LIFE_PATH)

Saved: data\life_expectancy_by_local_authority.csv (22.7 MB)


## Step 6 – First look at the life expectancy data
**Look, don't clean yet.** Cleaning is next week.

In [2]:
import pandas as pd
life = pd.read_csv(LIFE_PATH)
print("Rows, columns:", life.shape)
print("Columns:", list(life.columns))
life.head()

NameError: name 'LIFE_PATH' is not defined

In [ ]:

# How many different values does each column have?
life.nunique().sort_values()

sex                            2
Sex                            2
Time                          17
two-year-intervals            17
age-groups                    20
AgeGroups                     20
Geography                    420
administrative-geography     421
v4_2                        8341
Lower CI                    8343
Upper CI                    8352
dtype: int64